In [1]:
import os

SRC_DIR = "/kaggle/input/datasets/swetha344/pyg-210-cu128-wheels"
DST_DIR = "/kaggle/working/fixed_wheels"

os.makedirs(DST_DIR, exist_ok=True)

rename_map = {
    "torch_scatter-2.1.2pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_cluster-1.6.3pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "pyg_lib-0.4.0pt210cu128-cp312-cp312-linux_x86_64.whl": "pyg_lib-0.4.0+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_geometric-2.7.0-py3-none-any.whl": "torch_geometric-2.7.0-py3-none-any.whl",
    "ogb-1.3.6-py3-none-any.whl": "ogb-1.3.6-py3-none-any.whl"
}

for old_name, new_name in rename_map.items():
    src = os.path.join(SRC_DIR, old_name)
    dst = os.path.join(DST_DIR, new_name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

!pip install --no-index --no-deps --force-reinstall /kaggle/working/fixed_wheels/*.whl

Processing ./fixed_wheels/ogb-1.3.6-py3-none-any.whl
Processing ./fixed_wheels/torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl
Processing ./fixed_wheels/torch_geometric-2.7.0-py3-none-any.whl
Processing ./fixed_wheels/torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl


In [2]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [4]:
import os
import pandas as pd
import torch
from ogb.nodeproppred import PygNodePropPredDataset

# 1. Load the graph
dataset = PygNodePropPredDataset(name='ogbn-arxiv')
data = dataset[0]

# 2. Load Mapping Files
# The 'nodeidx2paperid' file maps: Node Index -> MAG Paper ID
path = os.path.join(dataset.root, 'mapping', 'nodeidx2paperid.csv.gz')
nodeidx2paperid = pd.read_csv(path)

# The 'titleabs' file contains: MAG Paper ID -> Title/Abstract
if not os.path.exists('titleabs.tsv'):
    !wget https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
    !gunzip titleabs.tsv.gz

titleabs = pd.read_csv('titleabs.tsv', sep='\t', names=['paper id', 'title', 'abstract'])

nodeidx2paperid['paper id'] = pd.to_numeric(nodeidx2paperid['paper id'], errors='coerce')
titleabs['paper id'] = pd.to_numeric(titleabs['paper id'], errors='coerce')
nodeidx2paperid = nodeidx2paperid.dropna(subset=['paper id']).astype({'paper id': 'int64'})
titleabs = titleabs.dropna(subset=['paper id']).astype({'paper id': 'int64'})

print("Aligning text to node indices...")
# We merge on nodeidx2paperid so the final dataframe order matches the Graph Node Indices (0 to 169,342)
df = pd.merge(nodeidx2paperid, titleabs, on='paper id', how='left')

# Handle missing text (if any papers in the graph aren't in the tsv)
df['title'] = df['title'].fillna('Unknown Title')
df['abstract'] = df['abstract'].fillna('Unknown Abstract')
df['text'] = df['title'] + ". " + df['abstract']

texts = df['text'].tolist()
print(f"Successfully aligned {len(texts)} texts.")

/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


Aligning text to node indices...
Successfully aligned 169343 texts.


In [5]:
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model_text = AutoModel.from_pretrained('roberta-base').to(device)
model_text.eval()

def get_embeddings(text_list):
    text_list = [str(t) if t is not None else "" for t in text_list]
    inputs = tokenizer(text_list, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_text(**inputs)
        # Use mean pooling for more stable GCN features
        emb = outputs.last_hidden_state.mean(dim=1)
    return emb.cpu()

batch_size = 32
all_embeddings = []

print("Extracting RoBERTa features (this takes time)...")
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    try:
        all_embeddings.append(get_embeddings(batch))
    except Exception as e:
        print(f"Error at index {i}: {e}")
        # Fallback for broken rows: zero vector
        all_embeddings.append(torch.zeros((len(batch), 768)))

roberta_features = torch.cat(all_embeddings, dim=0)
torch.save(roberta_features, 'arxiv_roberta_features.pt')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extracting RoBERTa features (this takes time)...


100%|██████████| 5292/5292 [1:08:29<00:00,  1.29it/s]


In [15]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

# 1. Prepare Graph
data.edge_index = to_undirected(data.edge_index)
data.x = torch.load('ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)
data = data.to(device)

split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)
valid_idx = split_idx['valid'].to(device)
test_idx = split_idx['test'].to(device)

# 2. Define Model
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        self.convs.append(GCNConv(hidden_channels, out_channels))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index).log_softmax(dim=-1)

# 3. Multi-Run Execution
all_results = []
for run in range(10):
    seed = run 
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/10 (Seed: {seed})")
    
    model = GCN(768, 256, dataset.num_classes, 3, 0.5).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
    
    best_val, best_test = 0, 0
    for epoch in range(1, 501):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                preds = logits.argmax(dim=-1, keepdim=True)
                val_acc = (preds[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (preds[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)
                
                if val_acc > best_val:
                    best_val, best_test = val_acc, test_acc
            print(f"Run {run+1} | Epoch {epoch} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")
            
    all_results.append(best_test)
    print(f"Run {run+1} Finished. Best Test Acc: {best_test:.4f}")

print(f"\nFinal Test Accuracy: {np.mean(all_results):.4f} ± {np.std(all_results):.4f}")

/tmp/ipykernel_55/3421593909.py:8: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  data.x = torch.load('ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)



>>> Starting Run 1/10 (Seed: 0)
Run 1 | Epoch 20 | Val: 0.5257 | Test: 0.5251
Run 1 | Epoch 40 | Val: 0.6742 | Test: 0.6678
Run 1 | Epoch 60 | Val: 0.7148 | Test: 0.7105
Run 1 | Epoch 80 | Val: 0.7230 | Test: 0.7223
Run 1 | Epoch 100 | Val: 0.7233 | Test: 0.7174
Run 1 | Epoch 120 | Val: 0.7326 | Test: 0.7195
Run 1 | Epoch 140 | Val: 0.7269 | Test: 0.7232
Run 1 | Epoch 160 | Val: 0.7322 | Test: 0.7188
Run 1 | Epoch 180 | Val: 0.7200 | Test: 0.7105
Run 1 | Epoch 200 | Val: 0.7076 | Test: 0.7144
Run 1 | Epoch 220 | Val: 0.7262 | Test: 0.7144
Run 1 | Epoch 240 | Val: 0.7367 | Test: 0.7251
Run 1 | Epoch 260 | Val: 0.7196 | Test: 0.7028
Run 1 | Epoch 280 | Val: 0.7196 | Test: 0.7234
Run 1 | Epoch 300 | Val: 0.7216 | Test: 0.7138
Run 1 | Epoch 320 | Val: 0.7043 | Test: 0.6821
Run 1 | Epoch 340 | Val: 0.7290 | Test: 0.7153
Run 1 | Epoch 360 | Val: 0.7054 | Test: 0.7067
Run 1 | Epoch 380 | Val: 0.6751 | Test: 0.6840
Run 1 | Epoch 400 | Val: 0.7199 | Test: 0.7182
Run 1 | Epoch 420 | Val: 0.6742